# GeoCebada — visualización rápida de los datos

Notebook introductorio para explorar los datos oficiales sin necesidad de conocer la estructura interna del proyecto.

**Objetivos**

- instalar `geocebada` en modo editable desde la propia notebook;
- cargar `BASIC`, `PRO` y el split oficial;
- revisar tamaños, columnas, missingness y rendimiento;
- visualizar relaciones con un **pairplot triangular ligero**;
- construir una vista descriptiva por parcela para el ciclo abril–octubre de 2025;
- mostrar al final el diccionario `docs/VARIABLES.md`.

> Las tablas satelitales son longitudinales: una parcela aparece muchas veces. Los puntos de una captura **no son observaciones independientes** para inferencia o validación de modelos.


## 0. Instalación

Sí: una notebook puede ejecutar `pip install -e`. La celda siguiente busca automáticamente la raíz del repositorio y hace una instalación editable del paquete.

Esto significa que, si un compañero actualiza el repositorio, **no tiene que copiar funciones ni modificar `sys.path`**. Sólo vuelve a ejecutar esta celda si cambia de entorno de Python.


In [ ]:
import subprocess
import sys
from pathlib import Path


def find_repo_root(start: Path | None = None) -> Path:
    """Find the GeoCebada repository root from the current working directory."""
    start = (start or Path.cwd()).resolve()
    for candidate in (start, *start.parents):
        if (candidate / "pyproject.toml").exists() and (candidate / "src" / "geocebada").exists():
            return candidate
    raise RuntimeError(
        "No se encontró la raíz de GeoCebada. Abre Jupyter desde dentro del repositorio."
    )


ROOT = find_repo_root()
print(f"Repositorio: {ROOT}")

subprocess.check_call(
    [sys.executable, "-m", "pip", "install", "-e", str(ROOT)]
)

print("geocebada instalado en modo editable.")


## 1. Importaciones y configuración

Cambia `SOURCE` entre `"BASIC"` y `"PRO"` para explorar una u otra tabla satelital.

El pairplot usa una muestra aleatoria pequeña. Los puntos se dibujan con `rasterized=True`, de modo que si después guardan la figura en PDF/SVG los miles de puntos no convierten el archivo en un vector enorme.


In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import Markdown, display

from geocebada.data import (
    attach_yield_split_metadata,
    load_basic_data,
    load_pro_data,
    load_yield_split,
)

RANDOM_SEED = 42
SOURCE = "BASIC"  # "BASIC" o "PRO"
PAIR_SAMPLE = 2_000

plt.rcParams["figure.dpi"] = 110
pd.set_option("display.max_columns", 120)


## 2. Cargar los datos oficiales


In [ ]:
split = load_yield_split()
basic = load_basic_data()
pro = load_pro_data()

print(f"Split: {split.shape[0]:,} filas × {split.shape[1]:,} columnas")
print(f"BASIC: {basic.shape[0]:,} filas × {basic.shape[1]:,} columnas")
print(f"PRO:   {pro.shape[0]:,} filas × {pro.shape[1]:,} columnas")

display(split.head())


### Resumen rápido del split


In [ ]:
summary = (
    split.groupby("CONJUNTO", dropna=False)
    .agg(
        parcelas=("ID_POLIGONO", "nunique"),
        area_total_ha=("AREA_HA", "sum"),
        rendimiento_observado=("RENDIMIENTO_T_HA", "count"),
    )
)
display(summary)


In [ ]:
train = split.loc[split["CONJUNTO"].eq("ENTRENAMIENTO")].copy()

fig, axes = plt.subplots(1, 2, figsize=(10, 3.8))

axes[0].hist(train["RENDIMIENTO_T_HA"].dropna(), bins=14)
axes[0].set_title("Rendimiento — entrenamiento")
axes[0].set_xlabel("ton/ha")
axes[0].set_ylabel("Parcelas")

axes[1].scatter(
    train["AREA_HA"],
    train["RENDIMIENTO_T_HA"],
    s=24,
    alpha=0.7,
)
axes[1].set_title("Área vs rendimiento")
axes[1].set_xlabel("Área (ha)")
axes[1].set_ylabel("Rendimiento (ton/ha)")

fig.tight_layout()
plt.show()


## 3. Elegir BASIC o PRO

Se adjuntan `AREA_HA`, `CONJUNTO` y `RENDIMIENTO_T_HA` por `ID_POLIGONO`. Para las parcelas de `PREDICCION`, el rendimiento continúa vacío.


In [ ]:
if SOURCE.upper() == "BASIC":
    satellite = attach_yield_split_metadata(basic)
    default_pair_columns = [
        "ndvi_promedio",
        "evi_promedio",
        "lai_promedio",
        "ndwi_promedio",
        "msi_promedio",
        "nddi_promedio",
    ]
elif SOURCE.upper() == "PRO":
    satellite = attach_yield_split_metadata(pro)
    default_pair_columns = [
        "ndvi_promedio",
        "evi_promedio",
        "lai_promedio",
        "msavi_promedio",
    ]
else:
    raise ValueError("SOURCE debe ser 'BASIC' o 'PRO'.")

print(f"{SOURCE.upper()}: {satellite.shape[0]:,} filas × {satellite.shape[1]:,} columnas")
print(f"Parcelas únicas: {satellite['ID_POLIGONO'].nunique():,}")
print("Sensores:", satellite["sensor"].dropna().unique().tolist())

display(satellite.head())


### Missingness de las variables seleccionadas


In [ ]:
missing = (
    satellite[default_pair_columns]
    .isna()
    .mean()
    .mul(100)
    .sort_values(ascending=False)
    .rename("% missing")
    .to_frame()
)
display(missing)


## 4. Pairplot triangular ligero

Este pairplot es deliberadamente sencillo:

- sólo muestra la diagonal y el triángulo inferior;
- usa como máximo `PAIR_SAMPLE` filas completas;
- los puntos están rasterizados;
- no usa las 100 mil observaciones si no es necesario.

En esta primera figura cada punto es una **captura satelital**, no una parcela independiente.


In [ ]:
def triangular_pairplot(
    frame: pd.DataFrame,
    columns: list[str],
    *,
    sample: int = 2_000,
    random_state: int = 42,
    bins: int = 24,
) -> tuple[plt.Figure, np.ndarray]:
    """Draw a lightweight lower-triangular pairplot with rasterized scatter points."""
    data = frame[columns].dropna()

    if data.empty:
        raise ValueError("No hay filas completas para las columnas seleccionadas.")

    if len(data) > sample:
        data = data.sample(sample, random_state=random_state)

    n = len(columns)
    fig, axes = plt.subplots(
        n,
        n,
        figsize=(1.8 * n, 1.8 * n),
        squeeze=False,
    )

    for row, y_col in enumerate(columns):
        for col, x_col in enumerate(columns):
            ax = axes[row, col]

            if col > row:
                ax.axis("off")
                continue

            if row == col:
                ax.hist(data[x_col], bins=bins)
            else:
                ax.scatter(
                    data[x_col],
                    data[y_col],
                    s=7,
                    alpha=0.22,
                    linewidths=0,
                    rasterized=True,
                )

            if row == n - 1:
                ax.set_xlabel(x_col, fontsize=8, rotation=25, ha="right")
            else:
                ax.set_xticklabels([])

            if col == 0:
                ax.set_ylabel(y_col, fontsize=8)
            else:
                ax.set_yticklabels([])

            ax.tick_params(labelsize=7)

    fig.suptitle(
        f"Pairplot triangular — {len(data):,} observaciones muestreadas",
        y=1.01,
    )
    fig.tight_layout()
    return fig, axes


In [ ]:
pair_columns = [column for column in default_pair_columns if column in satellite.columns]

fig, _ = triangular_pairplot(
    satellite,
    pair_columns,
    sample=PAIR_SAMPLE,
    random_state=RANDOM_SEED,
)
plt.show()


## 5. Vista descriptiva por parcela para abril–octubre de 2025

El target oficial corresponde al ciclo **abril–octubre de 2025**. Para una visualización más cercana a la unidad final de predicción, aquí reducimos temporalmente las observaciones satelitales de ese periodo a **una fila por parcela** usando la mediana.

Esto es sólo una **vista descriptiva**. No constituye todavía el feature engineering final: faltan decisiones de nubosidad, sensor, ventanas temporales, agregaciones y validación.


In [ ]:
satellite_cycle = satellite.copy()
satellite_cycle["fecha_captura"] = pd.to_datetime(
    satellite_cycle["fecha_captura"],
    errors="coerce",
)

cycle_mask = satellite_cycle["fecha_captura"].between(
    "2025-04-01",
    "2025-10-31",
)

cycle = satellite_cycle.loc[cycle_mask].copy()

parcel_satellite = (
    cycle.groupby("ID_POLIGONO", as_index=False)[pair_columns]
    .median(numeric_only=True)
)

parcel_view = split.merge(
    parcel_satellite,
    on="ID_POLIGONO",
    how="left",
    validate="one_to_one",
)

print(f"Parcelas en la vista: {len(parcel_view):,}")
display(parcel_view.head())


### Pairplot a nivel de parcela

Ahora cada punto sí corresponde a una parcela. Para incluir `RENDIMIENTO_T_HA`, sólo se muestran parcelas de entrenamiento.


In [ ]:
parcel_train = parcel_view.loc[
    parcel_view["CONJUNTO"].eq("ENTRENAMIENTO")
].copy()

parcel_pair_columns = [
    "AREA_HA",
    *pair_columns[:5],
    "RENDIMIENTO_T_HA",
]
parcel_pair_columns = [
    column for column in parcel_pair_columns if column in parcel_train.columns
]

fig, _ = triangular_pairplot(
    parcel_train,
    parcel_pair_columns,
    sample=len(parcel_train),
    random_state=RANDOM_SEED,
)
plt.show()


## 6. Correlaciones descriptivas por parcela

La matriz siguiente es exploratoria. Correlación no implica causalidad y, con parcelas georreferenciadas, la independencia espacial tampoco debe darse por sentada.


In [ ]:
corr = parcel_train[parcel_pair_columns].corr(method="spearman")

fig, ax = plt.subplots(figsize=(8, 6))
image = ax.imshow(corr, vmin=-1, vmax=1)

ax.set_xticks(range(len(corr.columns)))
ax.set_xticklabels(corr.columns, rotation=45, ha="right", fontsize=8)
ax.set_yticks(range(len(corr.index)))
ax.set_yticklabels(corr.index, fontsize=8)

for row in range(len(corr.index)):
    for col in range(len(corr.columns)):
        ax.text(
            col,
            row,
            f"{corr.iloc[row, col]:.2f}",
            ha="center",
            va="center",
            fontsize=7,
        )

fig.colorbar(image, ax=ax, label="Spearman ρ")
ax.set_title("Correlación descriptiva — parcelas de entrenamiento")
fig.tight_layout()
plt.show()


## 7. Diccionario completo de variables

La siguiente celda muestra directamente `docs/VARIABLES.md`.

No copiamos el contenido dentro de la notebook: así el notebook siempre muestra la versión actual del diccionario cuando el equipo actualiza la documentación.


In [ ]:
variables_path = ROOT / "docs" / "VARIABLES.md"
display(Markdown(variables_path.read_text(encoding="utf-8")))


## Qué sigue

Esta notebook es para **entender rápidamente los datos**, no para entrenar el modelo final.

Las siguientes notebooks pueden separar responsabilidades:

1. auditoría temporal y de nubosidad;
2. feature engineering por parcela;
3. validación espacial/agrupada;
4. baselines y comparación de modelos;
5. explicabilidad y predicciones finales.
